# 03 · SEC: datos estructurados y sectores

**Objetivo:** descargar de la API oficial de la SEC la ficha de cada una de las 1.000 empresas (*submissions*): su actividad (código SIC) y la lista de documentos que ha presentado (*filings*). Con el código SIC se asigna a cada empresa uno de los **12 sectores de Fama-French**, la clasificación sectorial estándar en la investigación financiera. La lista de filings es la que usará el notebook 06 para localizar el texto que analizarán los modelos de sentimiento. Por último, se añaden unas pocas cifras financieras de 2025 de cada empresa como contexto.

**Entradas:**
- `data/cache/universo_recortado.parquet` (salida del notebook 02)
- `https://data.sec.gov/submissions/CIK##########.json` — una ficha por empresa, más las páginas con sus filings antiguos (todo cacheado en `data/cache/sec_submissions/`)
- `https://data.sec.gov/api/xbrl/frames/...` — cifras financieras de todas las empresas, una petición por cifra
- `Siccodes12.txt`, la tabla SIC → 12 sectores de la web de Kenneth French (cacheada en `data/cache/`)
- Variable `SEC_EDGAR_USER_AGENT` del `.env`

**Salidas:**
- `data/cache/empresas_sec.parquet` — una fila por empresa: SIC, sector, tipo de declarante y formulario anual que presenta
- `data/cache/filings_sec.parquet` — los informes 10-K, 10-Q, 8-K, 20-F, 40-F y 6-K de cada empresa desde 2016, con su fecha y documento principal
- `data/cache/cifras_financieras.parquet` — ingresos, beneficio neto, activos y patrimonio de 2025, con margen neto y ROE (API *frames*, cacheada en `data/cache/sec_frames/`)

In [1]:
import io
import time
import zipfile

import pandas as pd
import plotly.express as px
import requests
from dotenv import load_dotenv

from funciones import CARPETA_CACHE, SEC_TIMEOUT_SEGUNDOS, descargar_json_sec

load_dotenv();

## 1. Empresas de partida

Leemos de disco las 1.000 empresas del notebook 02. El CIK (10 dígitos) es la clave que usa la SEC para identificarlas.

In [2]:
empresas = pd.read_parquet(CARPETA_CACHE / "universo_recortado.parquet")
print("Empresas:", len(empresas))
empresas[["ticker", "cik", "name", "exchange"]].head()

Empresas: 1000


,ticker,cik,name,exchange
0,MU,0000723125,MICRON TECHNOLOGY INC,Nasdaq
1,NVDA,0001045810,NVIDIA CORP,Nasdaq
2,SNDK,0002023554,Sandisk Corp,Nasdaq
3,AAPL,0000320193,Apple Inc.,Nasdaq
4,TSLA,0001318605,"Tesla, Inc.",Nasdaq


## 2. ¿Qué devuelve la API de *submissions*? (respuesta real de Apple)

Antes de descargar 1.000 fichas miramos una completa. La ficha tiene dos partes:

- **Datos de la empresa:** nombre, código SIC (actividad), tipo de declarante, cierre del ejercicio fiscal…
- **`filings`:** la lista de documentos presentados. Viene en formato **columnar**: cada campo (`form`, `filingDate`…) es una lista y la posición *i* de todas las listas describe el mismo documento.

In [3]:
URL_SUBMISSIONS = "https://data.sec.gov/submissions/CIK{cik}.json"
CARPETA_SUBMISSIONS = CARPETA_CACHE / "sec_submissions"

ficha_apple = descargar_json_sec(
    URL_SUBMISSIONS.format(cik="0000320193"), ruta_cache=CARPETA_SUBMISSIONS / "CIK0000320193.json"
)

print("Claves de la ficha:", list(ficha_apple.keys()))
for campo in ["name", "sic", "sicDescription", "category", "entityType", "fiscalYearEnd", "stateOfIncorporation"]:
    print(f"  {campo:22} {ficha_apple[campo]!r}")
print("Campos de cada filing:", list(ficha_apple["filings"]["recent"].keys()))

Claves de la ficha: ['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings']
  name                   'Apple Inc.'
  sic                    '3571'
  sicDescription         'Electronic Computers'
  category               'Large accelerated filer'
  entityType             'operating'
  fiscalYearEnd          '0926'
  stateOfIncorporation   'CA'
Campos de cada filing: ['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber', 'items', 'core_type', 'size', 'isXBRL', 'isInlineXBRL', 'isXBRLNumeric', 'primaryDocument', 'primaryDocDescription']


In [4]:
filings_apple = pd.DataFrame(ficha_apple["filings"]["recent"])
print("Filings en 'recent':", len(filings_apple),
      "· desde", filings_apple["filingDate"].min(), "hasta", filings_apple["filingDate"].max())
print("Páginas adicionales con filings más antiguos:", ficha_apple["filings"]["files"])
print()
print("Tipos de documento más frecuentes:")
display(filings_apple["form"].value_counts().head(10))
filings_apple[["form", "filingDate", "reportDate", "accessionNumber", "primaryDocument", "items"]].head(8)

Filings en 'recent': 1001 · desde 2015-09-17 hasta 2026-10-08
Páginas adicionales con filings más antiguos: [{'name': 'CIK0000320193-submissions-001.json', 'filingCount': 1264, 'filingFrom': '1994-01-26', 'filingTo': '2015-09-15'}]

Tipos de documento más frecuentes:


form
4           598
8-K         102
144          49
424B2        42
10-Q         33
PX14A6G      27
SC 13G/A     22
FWP          21
DEFA14A      12
3            12
Name: count, dtype: int64

,form,filingDate,reportDate,accessionNumber,primaryDocument,items
0,4,2026-10-08,2026-10-06,0001140361-26-039158,xslF345X06/form4.xml,
1,4,2026-10-05,2026-10-01,0001140361-26-038674,xslF345X06/form4.xml,
2,4,2026-10-05,2026-10-01,0001140361-26-038672,xslF345X06/form4.xml,
3,4,2026-10-05,2026-10-01,0001140361-26-038669,xslF345X06/form4.xml,
4,4,2026-10-05,2026-10-01,0001140361-26-038667,xslF345X06/form4.xml,
5,144,2026-10-02,,0001958244-26-000636,xsl144X01/primary_doc.xml,
6,144,2026-10-02,,0001959173-26-007235,xsl144X01/primary_doc.xml,
7,144,2026-10-02,,0001969223-26-001055,xsl144X01/primary_doc.xml,


Observaciones sobre la respuesta real:

- La mayoría de documentos son **formularios 4 y 144** (compraventas de acciones por directivos), que no nos interesan para el sentimiento.
- La parte `recent` contiene al menos el último año o los últimos ~1.000 documentos; lo más antiguo está en páginas aparte (`files`). Para Apple, `recent` llega hasta 2015 porque presenta muchos documentos.
- En los **8-K**, el campo `items` indica qué tipo de hecho se comunica (p. ej. `2.02` = resultados, `5.02` = cambio de directivos).

Nos quedamos con los informes que contienen texto narrativo de la empresa:

| Formulario | Qué es | Quién lo presenta |
|---|---|---|
| 10-K | Informe anual | Empresas de EEUU |
| 10-Q | Informe trimestral | Empresas de EEUU |
| 8-K | Hecho relevante | Empresas de EEUU |
| 20-F | Informe anual | Empresas extranjeras |
| 40-F | Informe anual | Empresas canadienses (sistema simplificado) |
| 6-K | Hecho relevante / informe intermedio | Empresas extranjeras |

## 3. Descarga de las 1.000 fichas

Cada ficha se guarda en `data/cache/sec_submissions/CIK##########.json`, así que volver a ejecutar no repite peticiones. `descargar_json_sec` (en `funciones.py`) añade el User-Agent obligatorio y espera 0,1 s tras cada petición, para no superar el límite de 10 peticiones por segundo de la SEC. Si una descarga falla, se anota y se informa al final, no se ignora.

In [5]:
fichas = {}
errores_descarga = {}
inicio_descarga = time.time()

for numero, cik in enumerate(empresas["cik"], start=1):
    try:
        fichas[cik] = descargar_json_sec(URL_SUBMISSIONS.format(cik=cik), ruta_cache=CARPETA_SUBMISSIONS / f"CIK{cik}.json")
    except RuntimeError as error:
        errores_descarga[cik] = str(error)
    if numero % 200 == 0:
        print(f"  {numero}/{len(empresas)} ({time.time() - inicio_descarga:.0f} s)")

print("Fichas obtenidas:", len(fichas), "· Errores:", len(errores_descarga))
errores_descarga

  200/1000 (0 s)


  400/1000 (0 s)
  600/1000 (0 s)


  800/1000 (0 s)


  1000/1000 (0 s)
Fichas obtenidas: 1000 · Errores: 0


{}

## 4. Ficha de cada empresa

De cada ficha nos quedamos con los datos que describen a la empresa:

- **`sic` / `sicDescription`:** código de actividad (*Standard Industrial Classification*) de 4 dígitos.
- **`category`:** tipo de declarante según su tamaño (p. ej. *Large accelerated filer*).
- **`entityType`:** `operating` para empresas con actividad; otros valores indicarían vehículos de inversión.
- **`fiscalYearEnd`:** mes y día de cierre del ejercicio (Apple: `0926`, 26 de septiembre).

In [6]:
filas_empresa = []
for cik, ficha in fichas.items():
    filas_empresa.append({
        "cik": cik,
        "sic": ficha["sic"],
        "sic_descripcion": ficha["sicDescription"],
        # La SEC separa varias categorías con la etiqueta HTML "<br>": la cambiamos por una coma
        "categoria": ficha["category"].replace("<br>", ", ").strip(", ") or "Sin dato",
        "tipo_entidad": ficha["entityType"],
        "cierre_ejercicio": ficha["fiscalYearEnd"],
    })

empresas_sec = empresas[["ticker", "cik", "name"]].merge(pd.DataFrame(filas_empresa), on="cik", how="left")

print("Tipo de entidad:")
display(empresas_sec["tipo_entidad"].value_counts(dropna=False))
print("Categoría de declarante:")
display(empresas_sec["categoria"].value_counts(dropna=False))
print("Empresas sin código SIC:", (empresas_sec["sic"].fillna("") == "").sum())

Tipo de entidad:


tipo_entidad
operating    891
other        109
Name: count, dtype: int64

Categoría de declarante:


categoria
Large accelerated filer                                                      928
Sin dato                                                                      22
Non-accelerated filer                                                         21
Emerging growth company                                                       12
Non-accelerated filer, Emerging growth company                                 5
Accelerated filer                                                              5
Non-accelerated filer, Smaller reporting company                               3
Large Accelerated                                                              2
Non-accelerated filer, Smaller reporting company, Emerging growth company      1
Accelerated filer, Emerging growth company                                     1
Name: count, dtype: int64

Empresas sin código SIC: 0


- **`tipo_entidad = other`** corresponde a las **empresas extranjeras** (TSM, ASML, Alibaba, Novo Nordisk…): la SEC las registra como *foreign private issuers*. Siguen siendo empresas con actividad.
- La gran mayoría son *Large accelerated filers* (las de mayor tamaño), coherente con el suelo de capitalización de 2.000 M$. Las que no tienen categoría suelen ser extranjeras, a las que no se les aplica esta clasificación.

## 5. Sector: del código SIC a los 12 sectores de Fama-French

El código SIC tiene cientos de valores (Apple es el 3571, *Electronic Computers*): demasiado detallado para comparar grupos. **Kenneth French** publica una tabla que agrupa los códigos SIC en **12 sectores**, muy usada en investigación financiera. Usarla tiene dos ventajas:

1. **Es un estándar externo:** no hay que inventar ni defender una agrupación propia.
2. **Sale de un dato oficial de la SEC (el SIC)**, sin depender de la clasificación propia de Yahoo.

Limitación conocida: el sector 12 (*Other*) es un cajón de sastre (minería, construcción, transporte, hoteles, servicios a empresas…).

Primero miramos el fichero tal cual lo publica French:

In [7]:
URL_FAMA_FRENCH_12 = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Siccodes12.zip"
ruta_fama_french = CARPETA_CACHE / "Siccodes12.txt"

if not ruta_fama_french.exists():
    try:
        respuesta = requests.get(URL_FAMA_FRENCH_12, timeout=SEC_TIMEOUT_SEGUNDOS)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as error:
        raise RuntimeError(f"No se pudo descargar la tabla de Fama-French: {error}") from error
    with zipfile.ZipFile(io.BytesIO(respuesta.content)) as comprimido:
        ruta_fama_french.write_bytes(comprimido.read("Siccodes12.txt"))

texto_fama_french = ruta_fama_french.read_text(encoding="latin-1")
print(texto_fama_french[:700])

 1 NoDur  Consumer Nondurables -- Food, Tobacco, Textiles, Apparel, Leather, Toys
          0100-0999
          2000-2399
          2700-2749
          2770-2799
          3100-3199
          3940-3989

 2 Durbl  Consumer Durables -- Cars, TVs, Furniture, Household Appliances
          2500-2519
          2590-2599
          3630-3659
          3710-3711
          3714-3714
          3716-3716
          3750-3751
          3792-3792
          3900-3939
          3990-3999

 3 Manuf  Manufacturing -- Machinery, Trucks, Planes, Off Furn, Paper, Com Printing
          2520-2589
          2600-2699
          2750-2769
          3000-3099
          3200-3569
          3580-3629
          3700-370


El formato es: una línea con el número, el código corto y la descripción del sector, seguida de los **rangos de códigos SIC** que incluye (`0100-0999` = del 100 al 999). El sector 12 (*Other*) no tiene rangos: recoge todo lo que no encaja en los otros 11.

In [8]:
def leer_tabla_fama_french(texto: str) -> tuple[list[tuple[int, int, str]], dict[str, str]]:
    """Lee la tabla SIC -> sector de Fama-French.

    Devuelve la lista de rangos (sic_inicio, sic_fin, sector) y un
    diccionario con la descripción de cada sector.
    """
    rangos, descripciones, sector_actual = [], {}, None
    for linea in texto.splitlines():
        partes = linea.split()
        if not partes:
            continue
        if "-" in partes[0]:                      # línea de rango: "0100-0999"
            inicio, fin = partes[0].split("-")
            rangos.append((int(inicio), int(fin), sector_actual))
        else:                                     # línea de sector: " 1 NoDur  Consumer Nondurables -- ..."
            sector_actual = partes[1]
            descripciones[sector_actual] = " ".join(partes[2:])
    return rangos, descripciones


def asignar_sector(sic: str, rangos: list[tuple[int, int, str]]) -> str:
    """Devuelve el sector de Fama-French de un código SIC; 'Other' si no cae en ningún rango."""
    if not sic:
        return "Sin SIC"
    for inicio, fin, sector in rangos:
        if inicio <= int(sic) <= fin:
            return sector
    return "Other"


rangos_sectores, descripcion_sectores = leer_tabla_fama_french(texto_fama_french)
print("Rangos leídos:", len(rangos_sectores), "· Sectores:", list(descripcion_sectores))

Rangos leídos: 49 · Sectores: ['NoDur', 'Durbl', 'Manuf', 'Enrgy', 'Chems', 'BusEq', 'Telcm', 'Utils', 'Shops', 'Hlth', 'Money', 'Other']


In [9]:
empresas_sec["sector"] = empresas_sec["sic"].fillna("").map(lambda sic: asignar_sector(sic, rangos_sectores))
empresas_sec["sector_descripcion"] = empresas_sec["sector"].map(descripcion_sectores)

# Comprobación a mano con empresas conocidas
empresas_sec.set_index("ticker").loc[["AAPL", "NVDA", "JPM", "XOM", "LLY", "WMT", "TSLA", "NEE"], ["name", "sic", "sic_descripcion", "sector"]]

,name,sic,sic_descripcion,sector
ticker,,,,
AAPL,Apple Inc.,3571,Electronic Computers,BusEq
NVDA,NVIDIA CORP,3674,Semiconductors & Related Devices,BusEq
JPM,JPMORGAN CHASE & CO,6021,National Commercial Banks,Money
XOM,ExxonMobil Holdings Corp,2911,Petroleum Refining,Enrgy
LLY,ELI LILLY & Co,2834,Pharmaceutical Preparations,Hlth
WMT,Walmart Inc.,5331,Retail-Variety Stores,Shops
TSLA,"Tesla, Inc.",3711,Motor Vehicles & Passenger Car Bodies,Durbl
NEE,NEXTERA ENERGY INC,4911,Electric Services,Utils


In [10]:
conteo_sectores = empresas_sec["sector"].value_counts()
grafico = px.bar(
    x=conteo_sectores.index, y=conteo_sectores.values,
    labels={"x": "Sector (Fama-French 12)", "y": "Nº de empresas"},
    title="Empresas por sector",
)
grafico.show(renderer="notebook_connected")
conteo_sectores.to_frame("empresas").join(pd.Series(descripcion_sectores, name="descripción"))

,empresas,descripción
sector,,
BusEq,217,"Business Equipment -- Computers, Software, and..."
Money,183,Finance
Other,155,"Other -- Mines, Constr, BldMt, Trans, Hotels, ..."
Manuf,101,"Manufacturing -- Machinery, Trucks, Planes, Of..."
Hlth,90,"Healthcare, Medical Equipment, and Drugs"
Shops,82,"Wholesale, Retail, and Some Services (Laundrie..."
Utils,43,Utilities
NoDur,37,"Consumer Nondurables -- Food, Tobacco, Textile..."
Enrgy,36,"Oil, Gas, and Coal Extraction and Products"


¿Qué actividades caen en el cajón de sastre *Other*? Si es muy grande, conviene saber qué contiene:

In [11]:
empresas_sec.loc[empresas_sec["sector"] == "Other", "sic_descripcion"].value_counts().head(15)

sic_descripcion
Services-Business Services, NEC                            27
Gold and Silver Ores                                       15
Metal Mining                                               10
Hotels & Motels                                             8
Mining & Quarrying of  Nonmetallic Minerals (No Fuels)      6
Services-Commercial Physical & Biological Research          5
Air Transportation, Scheduled                               5
Railroads, Line-Haul Operating                              5
Water Transportation                                        4
Heavy Construction Other Than Bldg Const - Contractors      4
Trucking (No Local)                                         4
Operative Builders                                          4
Refuse Systems                                              4
Electrical Work                                             4
Services-Consumer Credit Reporting, Collection Agencies     4
Name: count, dtype: int64

## 6. Filings con texto narrativo

De cada empresa nos quedamos con los formularios de la tabla de la sección 2, su campo `items` y el documento principal (`primaryDocument`), que es el archivo que descargará el notebook 06.

**La parte `recent` no basta.** Solo trae los últimos ~1.000 documentos de cualquier tipo. Las empresas que presentan muchos (bancos, aseguradoras) se quedan sin histórico: de JPMorgan solo llega a octubre de 2025. Lo anterior está en **páginas adicionales** (`files`), cada una con su rango de fechas. Como el proyecto empieza en octubre de 2016, se descargan solo las páginas que llegan a 2016 o después, y de ellas se guardan los filings desde 2016.

In [12]:
FORMULARIOS_DE_INTERES = ["10-K", "10-Q", "8-K", "20-F", "40-F", "6-K"]

URL_PAGINA_SUBMISSIONS = "https://data.sec.gov/submissions/{nombre}"
COLUMNAS_FILING = ["form", "filingDate", "reportDate", "accessionNumber", "primaryDocument", "items"]


def filings_de_una_ficha(ficha: dict, cik: str, desde: str = "2016-01-01") -> pd.DataFrame:
    """Filings de interés de una ficha: la parte 'recent' más las páginas antiguas que llegan a 'desde'."""
    tablas = [pd.DataFrame(ficha["filings"]["recent"])[COLUMNAS_FILING]]
    for pagina in ficha["filings"]["files"]:
        if pagina["filingTo"] >= desde:
            datos_pagina = descargar_json_sec(URL_PAGINA_SUBMISSIONS.format(nombre=pagina["name"]),
                                              ruta_cache=CARPETA_SUBMISSIONS / pagina["name"])
            antiguos = pd.DataFrame(datos_pagina)[COLUMNAS_FILING]
            antiguos = antiguos[antiguos["filingDate"] >= desde]
            if len(antiguos) > 0:
                tablas.append(antiguos)
    todos = pd.concat(tablas, ignore_index=True)
    # Las páginas se solapan en el día en que acaba una y empieza la siguiente: el mismo filing sale dos veces
    todos = todos.drop_duplicates(subset="accessionNumber")
    todos = todos[todos["form"].isin(FORMULARIOS_DE_INTERES)].copy()
    todos.insert(0, "cik", cik)
    return todos


inicio_paginas = time.time()
tablas_filings = [filings_de_una_ficha(ficha, cik) for cik, ficha in fichas.items()]
print(f"Fichas y páginas antiguas leídas en {time.time() - inicio_paginas:.0f} s")

filings = pd.concat(tablas_filings, ignore_index=True)
filings = filings.rename(columns={
    "form": "formulario", "filingDate": "fecha_presentacion", "reportDate": "fecha_periodo",
    "accessionNumber": "numero_acceso", "primaryDocument": "documento_principal",
})
filings["fecha_presentacion"] = pd.to_datetime(filings["fecha_presentacion"])
filings["fecha_periodo"] = pd.to_datetime(filings["fecha_periodo"], errors="coerce")  # vacía en algunos 8-K/6-K

# Algunos filings muy antiguos (1996-2000, de la época del papel) no tienen documento principal: no se pueden descargar
sin_documento = filings["documento_principal"] == ""
print("Filings sin documento principal (se descartan):", sin_documento.sum(),
      "· años:", sorted(filings.loc[sin_documento, "fecha_presentacion"].dt.year.unique().tolist()))
filings = filings[~sin_documento].reset_index(drop=True)

print("Filings de interés:", len(filings))
filings["formulario"].value_counts()

Fichas y páginas antiguas leídas en 2 s
Filings sin documento principal (se descartan): 59 · años: [1996, 1997, 1998, 1999, 2000]
Filings de interés: 218178


formulario
8-K     118580
6-K      61240
10-Q     27436
10-K      9166
20-F      1221
40-F       535
Name: count, dtype: int64

### Caso especial: XOM, una empresa que cambió de CIK

En julio de 2026 Exxon se reorganizó bajo una sociedad *holding* nueva, y el ticker `XOM` apunta ahora a ese CIK nuevo, que no tiene histórico (sección 7). Los informes anteriores siguen bajo el CIK de **Exxon Mobil Corp** (`0000034088`, comprobado contra la SEC el 20 oct 2026).

Se añaden esos informes a XOM con una excepción escrita a mano. La columna `cik_archivo` guarda el CIK bajo el que está **guardado el documento** en la SEC, que es el que hace falta para descargarlo; para el resto de empresas coincide con `cik`.

In [13]:
CIK_ANTIGUO_EXXON = "0000034088"
cik_xom = empresas.loc[empresas["ticker"] == "XOM", "cik"].item()

filings["cik_archivo"] = filings["cik"]
ficha_exxon = descargar_json_sec(URL_SUBMISSIONS.format(cik=CIK_ANTIGUO_EXXON),
                                 ruta_cache=CARPETA_SUBMISSIONS / f"CIK{CIK_ANTIGUO_EXXON}.json")
print("Ficha antigua:", ficha_exxon["name"])

filings_exxon = filings_de_una_ficha(ficha_exxon, cik=cik_xom)
filings_exxon = filings_exxon.rename(columns={
    "form": "formulario", "filingDate": "fecha_presentacion", "reportDate": "fecha_periodo",
    "accessionNumber": "numero_acceso", "primaryDocument": "documento_principal",
})
filings_exxon["fecha_presentacion"] = pd.to_datetime(filings_exxon["fecha_presentacion"])
filings_exxon["fecha_periodo"] = pd.to_datetime(filings_exxon["fecha_periodo"], errors="coerce")
filings_exxon["cik_archivo"] = CIK_ANTIGUO_EXXON

# Tras la reorganización, algún informe se presentó a la vez con los dos CIK: se queda el del CIK nuevo
ya_en_xom = filings.loc[filings["cik"] == cik_xom, "numero_acceso"]
filings_exxon = filings_exxon[~filings_exxon["numero_acceso"].isin(ya_en_xom)]

filings = pd.concat([filings, filings_exxon], ignore_index=True)
print(f"Filings añadidos a XOM: {len(filings_exxon)} · 10-K desde {filings_exxon.loc[filings_exxon['formulario'] == '10-K', 'fecha_presentacion'].min().date()}")

Ficha antigua: EXXON MOBIL CORP
Filings añadidos a XOM: 197 · 10-K desde 2016-02-24


### ¿Qué formulario anual presenta cada empresa?

Es la pregunta clave para el notebook 07: las empresas de EEUU presentan **10-K** y las extranjeras **20-F** o **40-F**, con secciones distintas. Para cada empresa miramos qué formulario anual presentó más recientemente.

In [14]:
anuales = filings[filings["formulario"].isin(["10-K", "20-F", "40-F"])]
ultimo_anual = anuales.sort_values("fecha_presentacion").groupby("cik").last()

empresas_sec = empresas_sec.merge(
    ultimo_anual[["formulario", "fecha_presentacion"]].rename(columns={
        "formulario": "formulario_anual", "fecha_presentacion": "fecha_ultimo_anual"}),
    left_on="cik", right_index=True, how="left",
)
empresas_sec["formulario_anual"] = empresas_sec["formulario_anual"].fillna("ninguno")

print("Formulario anual más reciente de cada empresa:")
display(empresas_sec["formulario_anual"].value_counts())
print("Empresas sin informe anual en 'recent' (p. ej. salidas a bolsa recientes):")
empresas_sec.loc[empresas_sec["formulario_anual"] == "ninguno", ["ticker", "name", "sector"]]

Formulario anual más reciente de cada empresa:


formulario_anual
10-K       883
20-F        80
40-F        31
ninguno      6
Name: count, dtype: int64

Empresas sin informe anual en 'recent' (p. ej. salidas a bolsa recientes):


,ticker,name,sector
7,SPCX,SPACE EXPLORATION TECHNOLOGIES CORP,BusEq
19,SKHY,SK hynix Inc.,BusEq
79,CBRS,Cerebras Systems Inc.,BusEq
191,HONA,Honeywell Aerospace Inc.,Manuf
945,INIO,INNIO N.V.,Manuf
954,XE,"X-Energy, Inc.",Manuf


Lectura:

- **10-K para la gran mayoría**, y alrededor de un 11 % de extranjeras con **20-F** (europeas, asiáticas…) o **40-F** (canadienses). En el notebook 07 habrá que decidir si se adapta el parseo a estos formularios, cuyas secciones se llaman distinto, o si el sentimiento de filings se limita a los 10-K.
- **Sin informe anual:** salidas a bolsa recientes (SpaceX, Cerebras, X-Energy, INNIO), un nuevo programa de ADR (SK hynix), una escisión reciente (Honeywell Aerospace) y un caso especial: **XOM**. Su ticker apunta ahora a *ExxonMobil Holdings Corp*, una sociedad *holding* nueva con filings solo desde julio de 2026. El histórico de informes de Exxon sigue bajo el CIK de la sociedad anterior. Es un ejemplo de que **el CIK de una empresa puede cambiar** con una reorganización; sus filings antiguos se añadieron en la sección 6 con el CIK anterior.

### ¿Cuánta historia de filings tenemos?

Con las páginas antiguas, medimos por empresa la fecha del filing de interés más antiguo disponible. Esto condiciona la comparación entre noticias y filings del backtesting (notebook 09), que necesita filings desde octubre de 2016.

In [15]:
filing_mas_antiguo = filings.groupby("cik")["fecha_presentacion"].min()
anios_de_filings = (pd.Timestamp("2026-10-09") - filing_mas_antiguo).dt.days / 365.25

tramos = pd.cut(anios_de_filings, bins=[0, 1, 3, 5, 9.9, 40],
                labels=["< 1 año", "1–3 años", "3–5 años", "5–10 años", "10 años o más"])
print("Empresas según la antigüedad de su filing más antiguo:")
tramos.value_counts().sort_index()

Empresas según la antigüedad de su filing más antiguo:


fecha_presentacion
< 1 año           14
1–3 años          34
3–5 años          31
5–10 años        142
10 años o más    779
Name: count, dtype: int64

## 7. Cifras financieras de 2025 (API *frames* de la SEC)

Las empresas presentan sus estados financieros con **etiquetas XBRL**: cada cifra lleva una etiqueta estándar (p. ej. `NetIncomeLoss` = beneficio neto). La SEC ofrece dos formas de consultarlas:

- **`companyfacts`:** todas las cifras de una empresa a lo largo de su historia. Para Apple son 3,8 MB, unos **4 GB** para las 1.000 empresas.
- **`frames`:** **una cifra, de todas las empresas, en un periodo**. Por ejemplo, el beneficio neto de 2025 de todas las empresas de EEUU en una sola petición (~0,5 MB).

Como solo queremos unas pocas cifras de contexto, usamos `frames`: 5 peticiones en lugar de 1.000 ficheros. Las cifras elegidas son:

| Cifra | Etiqueta XBRL | Periodo |
|---|---|---|
| Ingresos | `Revenues` o, si no la usa, `RevenueFromContractWithCustomerExcludingAssessedTax` | Año 2025 (`CY2025`) |
| Beneficio neto | `NetIncomeLoss` | Año 2025 (`CY2025`) |
| Activos totales | `Assets` | Cierre de 2025 (`CY2025Q4I`, la *I* indica un saldo en una fecha) |
| Patrimonio neto | `StockholdersEquity` | Cierre de 2025 (`CY2025Q4I`) |

Los ingresos necesitan dos etiquetas porque las empresas no usan todas la misma: desde la norma contable ASC 606 (2018), muchas declaran los ingresos por contratos con clientes con la segunda. `CY2025` agrupa el ejercicio fiscal que más se solapa con el año natural 2025 (el de Apple termina en septiembre).

Primero, la respuesta real de una de ellas:

In [16]:
URL_FRAMES = "https://data.sec.gov/api/xbrl/frames/us-gaap/{concepto}/USD/{periodo}.json"
CARPETA_FRAMES = CARPETA_CACHE / "sec_frames"

frame_ejemplo = descargar_json_sec(
    URL_FRAMES.format(concepto="NetIncomeLoss", periodo="CY2025"),
    ruta_cache=CARPETA_FRAMES / "NetIncomeLoss_CY2025.json",
)
print("Claves:", list(frame_ejemplo.keys()))
print("Etiqueta:", frame_ejemplo["label"], "· periodo:", frame_ejemplo["ccp"], "· nº de empresas:", frame_ejemplo["pts"])
pd.DataFrame(frame_ejemplo["data"]).head()

Claves: ['taxonomy', 'tag', 'ccp', 'uom', 'label', 'description', 'pts', 'data']
Etiqueta: Net Income (Loss) Attributable to Parent · periodo: CY2025 · nº de empresas: 5683


,accn,cik,entityName,loc,start,end,val
0,0001140361-26-031078,1750,AAR CORP.,-,2025-06-01,2026-05-31,1.877000e+08
1,0001628280-26-010185,1800,Abbott Laboratories,US-IL,2025-01-01,2025-12-31,6.524000e+09
2,0001193125-26-102079,2098,ACME UNITED CORP,US-CT,2025-01-01,2025-12-31,1.018442e+07
3,0001140361-26-016248,2186,BK Technologies Corporation,-,2025-01-01,2025-12-31,1.353600e+07
4,0000002488-26-000018,2488,"ADVANCED MICRO DEVICES, INC.",US-CA,2024-12-29,2025-12-27,4.335000e+09


Cada fila es una empresa: su CIK (como número, sin ceros a la izquierda), el valor (`val`, en dólares), la fecha de cierre del periodo (`end`) y el filing del que sale (`accn`). Escribimos una función que descarga una cifra y la deja lista para unir con nuestras empresas:

In [17]:
def descargar_cifra(concepto: str, periodo: str) -> pd.DataFrame:
    """Descarga una cifra XBRL (us-gaap, en dólares) de todas las empresas para un periodo.

    Usa la caché data/cache/sec_frames/. Devuelve un DataFrame con cik
    (texto de 10 dígitos), valor y fin_periodo.
    """
    frame = descargar_json_sec(
        URL_FRAMES.format(concepto=concepto, periodo=periodo),
        ruta_cache=CARPETA_FRAMES / f"{concepto}_{periodo}.json",
    )
    cifra = pd.DataFrame(frame["data"])
    cifra["cik"] = cifra["cik"].astype(str).str.zfill(10)
    return cifra.rename(columns={"val": "valor", "end": "fin_periodo"})[["cik", "valor", "fin_periodo"]]


ingresos_total = descargar_cifra("Revenues", "CY2025")
ingresos_contratos = descargar_cifra("RevenueFromContractWithCustomerExcludingAssessedTax", "CY2025")
beneficio = descargar_cifra("NetIncomeLoss", "CY2025")
activos = descargar_cifra("Assets", "CY2025Q4I")
patrimonio = descargar_cifra("StockholdersEquity", "CY2025Q4I")

# Ingresos: la primera etiqueta si la empresa la usa; si no, la segunda
ingresos = pd.concat([ingresos_total, ingresos_contratos]).drop_duplicates(subset="cik", keep="first")
print("Empresas (de toda la SEC) con ingresos:", len(ingresos),
      f"({len(ingresos_total)} con Revenues + {len(ingresos) - len(ingresos_total)} solo con la etiqueta ASC 606)")

Empresas (de toda la SEC) con ingresos: 4381 (2251 con Revenues + 2130 solo con la etiqueta ASC 606)


Unimos las cuatro cifras a nuestras 1.000 empresas y calculamos dos ratios sencillos:

- **Margen neto** = beneficio neto / ingresos: cuánto de cada dólar vendido queda como beneficio.
- **Rentabilidad sobre el patrimonio (ROE)** = beneficio neto / patrimonio neto: cuánto gana la empresa por cada dólar de sus accionistas. Si el patrimonio es negativo o cero, el ratio no tiene sentido y se deja vacío.

In [18]:
cifras = empresas_sec[["ticker", "cik", "name", "sector", "formulario_anual"]].copy()
for nombre, tabla in [("ingresos", ingresos), ("beneficio_neto", beneficio),
                      ("activos", activos), ("patrimonio_neto", patrimonio)]:
    cifras = cifras.merge(tabla[["cik", "valor"]].rename(columns={"valor": nombre}), on="cik", how="left")

cifras["margen_neto"] = cifras["beneficio_neto"] / cifras["ingresos"]
patrimonio_positivo = cifras["patrimonio_neto"].where(cifras["patrimonio_neto"] > 0)
cifras["roe"] = cifras["beneficio_neto"] / patrimonio_positivo

# Comprobación a mano con empresas conocidas (en miles de millones de dólares)
columnas_dinero = ["ingresos", "beneficio_neto", "activos", "patrimonio_neto"]
muestra = cifras.set_index("ticker").loc[["AAPL", "NVDA", "JPM", "LLY", "WMT", "XOM", "TSM"]]
(muestra[columnas_dinero] / 1e9).round(1).join(muestra[["margen_neto", "roe"]].round(3))

,ingresos,beneficio_neto,activos,patrimonio_neto,margen_neto,roe
ticker,,,,,,
AAPL,416.2,112.0,379.3,88.2,0.269,1.270
NVDA,215.9,120.1,206.8,157.3,0.556,0.763
JPM,182.4,57.0,4424.9,362.4,0.312,0.157
LLY,65.2,20.6,112.5,26.5,0.317,0.778
WMT,713.2,21.9,284.7,99.6,0.031,0.220
XOM,NaN,NaN,449.0,259.4,NaN,NaN
TSM,NaN,NaN,NaN,NaN,NaN,NaN


### Cobertura: ¿para cuántas empresas tenemos cada cifra?

La API `frames` solo cubre la taxonomía contable de EEUU (`us-gaap`). Las empresas extranjeras que presentan con normas internacionales (IFRS, la mayoría de las de 20-F) **no aparecen**: no es un fallo, es una limitación de la fuente. Lo medimos por tipo de formulario anual y, para los ingresos, por sector.

In [19]:
cobertura = cifras.groupby("formulario_anual")[columnas_dinero].apply(lambda grupo: grupo.notna().mean() * 100).round(0)
cobertura.insert(0, "empresas", cifras["formulario_anual"].value_counts())
print("% de empresas con cada cifra, según su formulario anual:")
cobertura

% de empresas con cada cifra, según su formulario anual:


,empresas,ingresos,beneficio_neto,activos,patrimonio_neto
formulario_anual,,,,,
10-K,883,93.0,95.0,100.0,93.0
20-F,80,28.0,25.0,26.0,26.0
40-F,31,0.0,0.0,0.0,0.0
ninguno,6,0.0,0.0,83.0,83.0


In [20]:
def porcentaje_con_dato(serie: pd.Series) -> float:
    """Porcentaje de valores no vacíos de una serie."""
    return serie.notna().mean() * 100


solo_10k = cifras["formulario_anual"] == "10-K"
cobertura_sector = pd.DataFrame({
    "todas las empresas (%)": cifras.groupby("sector")["ingresos"].apply(porcentaje_con_dato),
    "solo las de 10-K (%)": cifras[solo_10k].groupby("sector")["ingresos"].apply(porcentaje_con_dato),
    "extranjeras (20-F/40-F)": cifras[cifras["formulario_anual"].isin(["20-F", "40-F"])].groupby("sector").size(),
}).fillna(0).round(0).sort_values("todas las empresas (%)")
print("Cobertura de ingresos por sector:")
cobertura_sector

Cobertura de ingresos por sector:


,todas las empresas (%),solo las de 10-K (%),extranjeras (20-F/40-F)
sector,,,
Enrgy,67.0,86.0,8
Other,77.0,92.0,32
NoDur,78.0,91.0,5
Money,79.0,89.0,23
Telcm,79.0,92.0,2
Durbl,82.0,100.0,4
Utils,86.0,88.0,1
Chems,88.0,100.0,3
Hlth,88.0,98.0,10


Lectura:

- **Con todas las empresas**, el sector con menos ingresos disponibles es **Energía**, pero sobre todo porque tiene muchas extranjeras (Shell, BP, petroleras canadienses), que esta API no cubre.
- **Solo con las de 10-K**, la cobertura está entre el 88 % y el 100 % en todos los sectores. En número de empresas, el mayor hueco está en las **finanzas** (`Money`, 18 empresas sin ingresos): grandes bancos como Goldman Sachs, Wells Fargo o Morgan Stanley no tienen "ventas" y declaran sus ingresos con otras etiquetas (intereses, comisiones). Para ellos el beneficio, los activos y el patrimonio sí están disponibles.
- Estas cifras son **contexto** para la interfaz; no intervienen en las señales ni en el sentimiento. Por eso se acepta la cobertura incompleta en lugar de buscar etiquetas específicas para cada tipo de empresa.

## 8. Guardar las salidas

In [21]:
ruta_empresas_sec = CARPETA_CACHE / "empresas_sec.parquet"
ruta_filings = CARPETA_CACHE / "filings_sec.parquet"
ruta_cifras = CARPETA_CACHE / "cifras_financieras.parquet"

empresas_sec.to_parquet(ruta_empresas_sec, index=False)
filings.to_parquet(ruta_filings, index=False)
cifras.to_parquet(ruta_cifras, index=False)
print("Guardado:", ruta_empresas_sec.name, "-", len(empresas_sec), "filas")
print("Guardado:", ruta_filings.name, "-", len(filings), "filas")
print("Guardado:", ruta_cifras.name, "-", len(cifras), "filas")

Guardado: empresas_sec.parquet - 1000 filas
Guardado: filings_sec.parquet - 218375 filas
Guardado: cifras_financieras.parquet - 1000 filas


## Comprobaciones

In [22]:
# Todas las fichas descargadas y una fila por empresa
assert len(errores_descarga) == 0, f"Fallaron {len(errores_descarga)} descargas"
assert len(empresas_sec) == len(empresas) == 1_000
assert empresas_sec["cik"].is_unique

# Cada ficha corresponde a la empresa pedida
assert all(int(ficha["cik"]) == int(cik) for cik, ficha in fichas.items())

# Sectores: todos asignados y dentro de los 12 de Fama-French
assert set(empresas_sec["sector"]) <= set(descripcion_sectores) | {"Sin SIC"}
assert empresas_sec.loc[empresas_sec["ticker"] == "AAPL", "sector"].item() == "BusEq"
assert empresas_sec.loc[empresas_sec["ticker"] == "JPM", "sector"].item() == "Money"

# Filings: solo los formularios de interés, sin repetidos y de empresas del universo
assert set(filings["formulario"]) <= set(FORMULARIOS_DE_INTERES)
# Un mismo filing puede aparecer en dos empresas si lo presentaron juntas (p. ej. Exelon y Constellation
# en su escisión de 2022), así que la clave única es la pareja empresa + número de acceso
assert not filings.duplicated(subset=["cik", "numero_acceso"]).any()
assert set(filings["cik"]) <= set(empresas["cik"])
assert (filings["documento_principal"].fillna("") != "").all()

# Las páginas antiguas recuperan el histórico: JPM tiene 10-K de 2018 y XOM tiene 10-K anteriores a 2020
assert ((filings["cik"] == empresas.loc[empresas["ticker"] == "JPM", "cik"].item())
        & (filings["formulario"] == "10-K") & (filings["fecha_presentacion"].dt.year == 2018)).any()
assert ((filings["cik"] == cik_xom) & (filings["formulario"] == "10-K") & (filings["fecha_presentacion"] < "2020-01-01")).any()
assert filings["cik_archivo"].notna().all()

# Cifras financieras: una fila por empresa y valores coherentes
assert len(cifras) == 1_000 and cifras["cik"].is_unique
assert (cifras["activos"].dropna() > 0).all()
assert (cifras["ingresos"].dropna() >= 0).all()
assert cifras.loc[cifras["formulario_anual"] == "10-K", "activos"].notna().mean() > 0.9, "Cobertura de activos muy baja en 10-K"
assert cifras.loc[cifras["ticker"] == "AAPL", "ingresos"].item() > 100e9

print("Todas las comprobaciones OK")

Todas las comprobaciones OK
